# AI Chatbot Product Performance & User Engagement Analysis

**Product:** a customer-support AI chatbot (web, mobile, Slack). **Role:** product analyst.

**Questions**
1. **Adoption:** do signups start using the bot, and is usage growing?
2. **Task completion:** how often does the bot resolve the user's problem, and where does it fail?
3. **Repeat usage:** does a bad first experience cost us the user?
4. **Satisfaction:** what drives CSAT, and how reliable is the survey?
5. **Feedback themes:** what do users complain about? (LLM text classification)
6. **Recommendations:** where should the product team invest first?

> **Data note:** the data is **simulated** (`src/generate_data.py`, seed 42) because no public dataset combines completion,
> latency, CSAT and feedback text for a chatbot. The analysis workflow is what transfers to real data.
>
> Setup: `python src/generate_data.py && python src/load_data.py && python src/classify_feedback.py`

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from db import run_sql_file, get_connection

sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format
IMG = ROOT / "images"; IMG.mkdir(exist_ok=True)
def save(fig, name):
    fig.tight_layout(); fig.savefig(IMG / f"{name}.png", dpi=150, bbox_inches="tight")

## 1. Data quality

In [ ]:
dq = run_sql_file("01_data_quality_checks.sql")
for k, v in dq.items():
    print("---", k); display(v)

CSAT is a voluntary survey, so missing scores are **non-response**, not errors. We check for response bias in section 5.

## 2. Adoption and engagement

In [ ]:
adopt = run_sql_file("02_adoption_engagement.sql")
display(adopt["activation_rate_by_signup_month"])
wau = adopt["weekly_active_users"]
nr = adopt["new_vs_returning_users_weekly"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(wau["week"], wau["active_users"], marker="o")
axes[0].set_title("Weekly active users"); axes[0].tick_params(axis="x", rotation=90)
axes[1].bar(nr["week"], nr["new_users"], label="New", color="#4C72B0")
axes[1].bar(nr["week"], nr["returning_users"], bottom=nr["new_users"], label="Returning", color="#DD8452")
axes[1].set_title("New vs returning active users"); axes[1].tick_params(axis="x", rotation=90); axes[1].legend()
save(fig, "01_wau_new_vs_returning"); plt.show()
display(adopt["dau_mau_stickiness"]); display(adopt["usage_by_plan"])

## 3. Task completion and failure analysis

In [ ]:
tc = run_sql_file("03_task_completion.sql")
display(tc["overall_completion"])
by_intent = tc["completion_by_intent"]
lat = tc["completion_by_latency_bucket"]
fb_ = tc["completion_by_fallbacks"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sns.barplot(data=by_intent, y="intent", x="completion_pct", color="#4C72B0", ax=axes[0])
axes[0].axvline(float(tc["overall_completion"]["task_completion_pct"].iloc[0]), color="red", ls="--", label="overall"); axes[0].legend()
axes[0].set_title("Completion % by intent")
sns.barplot(data=lat, x="latency_bucket", y="completion_pct", color="#55A868", ax=axes[1]); axes[1].set_title("Completion % by response latency")
sns.barplot(data=fb_, x="fallbacks", y="completion_pct", color="#C44E52", ax=axes[2]); axes[2].set_title("Completion % by # fallbacks")
save(fig, "02_completion_drivers"); plt.show()
display(tc["failed_vs_successful_profile"])

In [ ]:
# Incident detection: days where latency is far above the previous 7-day average
d = tc["daily_latency_with_moving_avg"].dropna(subset=["prev_7d_avg_latency_ms"]).copy()
d["ratio"] = d["avg_latency_ms"] / d["prev_7d_avg_latency_ms"]
incidents = d[d["ratio"] > 1.5]
display(incidents)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pd.to_datetime(d["day"]), d["avg_latency_ms"], label="Daily avg latency (ms)")
ax.plot(pd.to_datetime(d["day"]), d["prev_7d_avg_latency_ms"], ls="--", label="Previous 7-day avg")
for day in incidents["day"]:
    ax.axvline(pd.to_datetime(day), color="red", alpha=.25)
ax.set_title("Response latency with detected incident days"); ax.legend()
save(fig, "03_latency_incident"); plt.show()

display(tc["weakest_intent_platform_combos"])

### Which factors really drive completion? (controlling for confounders)
Technical issues are both slow **and** hard, so raw latency buckets overstate the effect of speed.
A logistic regression separates the effects.

In [ ]:
con = get_connection()
df = pd.read_sql_query("""
    SELECT c.*, u.plan, u.platform
    FROM conversations c JOIN users u ON u.user_id = c.user_id
""", con)
df["latency_s"] = df["avg_response_latency_ms"] / 1000

model = smf.logit("resolved ~ latency_s + fallback_count + C(intent, Treatment('order_status')) + C(platform) + C(plan)", data=df).fit(disp=0)
odds = pd.DataFrame({"odds_ratio": np.exp(model.params), "ci_low": np.exp(model.conf_int()[0]),
                     "ci_high": np.exp(model.conf_int()[1]), "p_value": model.pvalues}).drop("Intercept")
odds.index = [i.replace("C(intent, Treatment('order_status'))[T.", "intent=").replace("C(platform)[T.", "platform=").replace("C(plan)[T.", "plan=").replace("]", "") for i in odds.index]
display(odds.sort_values("odds_ratio"))
print("Reading: odds ratio < 1 lowers the odds of task completion. E.g. each extra second of latency or each fallback multiplies the odds by the value shown.")

## 4. Repeat usage: does the first conversation decide retention?

In [ ]:
rep = run_sql_file("04_repeat_usage.sql")
first = rep["first_conversation_outcome_vs_return"]; display(first)
display(rep["first_conversation_latency_vs_return"])
display(rep["first_conversation_intent_vs_return"])

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=first, x="first_chat_outcome", y="return_within_14d_pct", palette=["#55A868", "#C44E52"], ax=ax)
for p in ax.patches: ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width()/2, p.get_height()), ha="center", va="bottom")
ax.set_title("Returned within 14 days, by first-chat outcome"); ax.set_xlabel(""); ax.set_ylabel("% of users")
save(fig, "04_first_chat_return"); plt.show()

# is the difference statistically significant?
x = first.set_index("first_chat_outcome")
n = x["users"]; ret = (x["users"] * x["return_within_14d_pct"] / 100).round().astype(int)
table = np.array([[ret.iloc[0], n.iloc[0] - ret.iloc[0]], [ret.iloc[1], n.iloc[1] - ret.iloc[1]]])
chi2, p, _, _ = stats.chi2_contingency(table)
print("Chi-square test of independence: p-value", "< 0.001" if p < 0.001 else f"= {p:.3f}")

In [ ]:
coh = rep["weekly_cohort_retention"]
heat = coh.pivot(index="cohort_month", columns="week_number", values="retention_pct").loc[:, 1:8]
fig, ax = plt.subplots(figsize=(9, 3.8))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="YlGnBu", cbar_kws={"label": "% of cohort active"}, ax=ax)
ax.set_title("Weekly retention by signup cohort"); ax.set_xlabel("Weeks since first conversation")
save(fig, "05_cohort_retention"); plt.show()

## 5. Customer satisfaction (CSAT)

In [ ]:
cs = run_sql_file("05_csat_analysis.sql")
display(cs["csat_summary"]); display(cs["survey_response_bias"]); display(cs["csat_by_intent"]); display(cs["csat_after_escalation"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=cs["csat_by_intent"], y="intent", x="avg_csat", color="#8172B2", ax=axes[0]); axes[0].set_title("Average CSAT by intent"); axes[0].set_xlim(1, 5)
sw = cs["csat_weekly_trend"]; sw = sw[sw["responses"] >= 30]
axes[1].plot(sw["week"], sw["avg_csat"], marker="o"); axes[1].tick_params(axis="x", rotation=90); axes[1].set_title("Weekly average CSAT")
save(fig, "06_csat"); plt.show()

**Survey bias caveat:** unresolved users answer the survey more often than resolved ones, so raw CSAT understates how satisfied
the *whole* user base is. Report the response rate next to every CSAT number, and prefer completion rate as the primary quality metric.

## 6. Feedback themes with an LLM text classifier
`src/classify_feedback.py` sends feedback to the **Claude API** in batches of 25 with a fixed label set and `temperature=0`
(JSON output, validated against the allowed themes). Without an API key it falls back to a keyword baseline so this notebook always runs.

In [ ]:
path = ROOT / "data" / "feedback_classified.csv"
if not path.exists():
    import subprocess; subprocess.run([sys.executable, str(ROOT / "src" / "classify_feedback.py")], check=True)
fb = pd.read_csv(path)
print("Classifier used:", fb["classifier"].iloc[0], "|", len(fb), "feedback items")

meta = pd.read_sql_query("SELECT conversation_id, intent, resolved, csat_score, avg_response_latency_ms, fallback_count FROM conversations", con)
fb = fb.merge(meta, on="conversation_id")

counts = fb["theme"].value_counts()
negative = fb[fb["theme"] != "positive_feedback"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
counts.sort_values().plot.barh(ax=axes[0], color="#4C72B0"); axes[0].set_title("All feedback by theme")
ct = pd.crosstab(negative["intent"], negative["theme"], normalize="index") * 100
sns.heatmap(ct, annot=True, fmt=".0f", cmap="Reds", ax=axes[1], cbar_kws={"label": "% of complaints"})
axes[1].set_title("Complaint themes by intent (row %)")
save(fig, "07_feedback_themes"); plt.show()

In [ ]:
# Evaluate the classifier against the simulation's ground-truth labels (only possible because the data is simulated)
truth = pd.read_csv(ROOT / "data" / "raw" / "feedback_ground_truth.csv")
ev = fb.merge(truth, on="conversation_id")
acc = (ev["theme"] == ev["true_theme"]).mean()
print(f"Overall accuracy vs ground truth: {acc:.1%}")

rows = []
for t in sorted(ev["true_theme"].unique()):
    tp = ((ev["theme"] == t) & (ev["true_theme"] == t)).sum()
    prec = tp / max((ev["theme"] == t).sum(), 1); rec = tp / max((ev["true_theme"] == t).sum(), 1)
    rows.append({"theme": t, "precision": prec, "recall": rec, "support": (ev["true_theme"] == t).sum()})
display(pd.DataFrame(rows))
print("Note: the simulated feedback is templated, so accuracy here is optimistic. Real feedback is messier, which is where an LLM beats keywords.")

## 7. Sizing the opportunity

In [ ]:
# Scenario: if we lift completion by 10 percentage points on the three weakest intents, how many more conversations resolve?
weak = by_intent.sort_values("completion_pct").head(3).copy()
weak["extra_resolved_at_+10pp"] = (weak["conversations"] * 0.10).round()
total = int(tc["overall_completion"]["conversations"].iloc[0]); base = tc["overall_completion"]["task_completion_pct"].iloc[0]
new_rate = base + 100 * weak["extra_resolved_at_+10pp"].sum() / total
display(weak[["intent", "conversations", "completion_pct", "extra_resolved_at_+10pp"]])
print(f"Overall task completion would move from {base:.1f}% to ~{new_rate:.1f}% (+{new_rate-base:.1f} pp).")
print("Second-order effect: users with a resolved first chat return far more often (section 4), so this also lifts retention.")

## 8. Findings and recommendations

> **Fill in with your own printed numbers.** Use the pattern: observation → evidence → impact → recommendation.

1. **Task completion is 59.5%; the weakest intents are `technical_issue` (25%), `refund_return` (37%) and `billing` (45%)** vs 81% for `order_status`.
   → Add tool integrations so the bot can *perform* refunds and billing changes instead of only explaining them.
2. **Speed and fallbacks hurt completion independently.** In the logistic regression (controlling for intent, platform and plan) each extra second of latency lowers the odds of completion by ~31% and each fallback by ~42%.
   → Set a latency budget with an alert when daily latency exceeds 1.5x its 7-day average (this flagged the 10-12 Apr incident, when completion fell to ~34%), and improve intent recognition and clarifying questions.
3. **The first conversation decides retention:** 72% of users with a resolved first chat returned within 14 days vs 25% of those with an unresolved first chat (chi-square p < 0.001).
   → Prioritise first-session quality: guided onboarding, route first-time users to the best-performing flows, proactive human handoff after 2 fallbacks.
4. **Top complaint themes** (LLM-classified, see section 6): lead with the biggest bars in the chart. → Map each theme to a roadmap item.
5. **CSAT has response bias:** only 35% respond, and unresolved users respond more (44% vs 30%). → Treat completion rate as the primary quality metric and add a one-tap thumbs up/down to raise coverage.

*(Numbers above come from the seed-42 simulation. Re-run to confirm yours match.)*

**Limitations:** simulated data; observational relationships (not causal); a controlled experiment (e.g. new fallback handling vs old) would confirm the impact.